In [29]:
## import os
os.makedirs("faces", exist_ok=True)

os.environ['PATH'] = (
    r"D:\Anaconda3\Lib\site-packages\nvidia\cudnn\bin"
    + ";" + os.environ.get('PATH', '')
)
import cv2
import time
import insightface
from IPython.display import clear_output
import contextlib
import numpy as np
RESOLUTIONS = {
    "1080p": (1920, 1080),
    "720p":  (1280, 720),
    "480p":  (854, 480),
    "360p":  (640, 360),
    "144p":  (256, 144),
}
ACCURACIES = {
    "low":    (160, 160),
    "medium": (224, 224),
    "high":   (320, 320),
    "max":    (640, 640),
}

def silent_model_load(provider, inference, accuracy):
    with open(os.devnull, "w") as f:
        with contextlib.redirect_stdout(f):
            with contextlib.redirect_stderr(f):
                app = insightface.app.FaceAnalysis(
                    name="buffalo_l",
                    allowed_modules=["detection","recognition"],
                    providers=provider
                )
                app.prepare(
                    ctx_id=inference,
                    det_size=ACCURACIES[accuracy]
                )
    return app
    
def should_exit(window_name):
    keyboard = (cv2.waitKey(1) & 0xFF) == ord("q")

    try:
        cross_window = cv2.getWindowProperty(window_name,cv2.WND_PROP_VISIBLE) < 1
    except:
        cross_window = True

    return keyboard or cross_window

def setup_camera(resolution="1080p"):
    cap = cv2.VideoCapture(0)
    w, h = RESOLUTIONS[resolution]

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    return cap

def load_model(processing = "GPU", accuracy="medium"):
    
    if processing == "GPU":
        provider = [
            "CUDAExecutionProvider",
            "CPUExecutionProvider"
        ]
        inference = 0
    else:
        provider = [
            "CPUExecutionProvider"
        ]
        inference = -1
    app = silent_model_load(provider, inference, accuracy)
    active_provider = app.models["detection"].session.get_providers()[0]
    clear_output(wait=True)
    print("=" * 40)
    print(f"Detector : buffalo_l")
    print(f"Provider : {active_provider}")
    print(f"Accuracy : {accuracy}")
    print(f"Frame Inf: {FRAME_SKIP}")
    print("=" * 40)
    return app
def draw_faces(
    frame,
    faces,
    database
):

    for face in faces:

        x1, y1, x2, y2 = map(
            int,
            face.bbox
        )

        name, score = recognize_face(
            face,
            database
        )
        if name == "Unknown":
            color = (0,0,255)      # Red
        else:
            color = (0,255,0)      # Green
            
        cv2.rectangle(
            frame,
            (x1,y1),
            (x2,y2),
            color,
            2
        )

        label = f"{name} ({score:.2f})"
    

        if name != "Unknown":
            label = f"{name} ({score:.2f})"
        
            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0,255,0),
                2
            )
def frame_calculation(frame, prev_time, frame_count, fps):
    #calculations
    frame_count += 1
    now = time.time()
    if now - prev_time >= 1:
        fps = frame_count
        frame_count = 0
        prev_time = now

    #display fps
    cv2.putText(
        frame,                    #img
        f"FPS: {fps}",            #text
        (10, 30),                 #org
        cv2.FONT_HERSHEY_SIMPLEX, #font
        1,                        #scale
        (0, 255, 0),              #color
        2)                        #thickness
    return prev_time, frame_count, fps
#functions of registeration----------------------------------
def register_menu():
    print("1. Live")
    print("2. Register")

    choice = input("Choose: ")

    if choice == "2":
        name = input("Enter name: ")
        return "register", name

    return "live", None

def registration_process(frame, faces):

    global register_started
    global register_start_time

    cv2.putText(
        frame,
        "Align your face",
        (10,60),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    center = draw_register_circle(frame)

    if len(faces) != 1:
        register_started = False
        return False

    if not face_in_circle(faces[0], center):
        register_started = False
        return False

    now = time.time()

    if not register_started:
        register_started = True
        register_start_time = now

    elapsed = now - register_start_time

    countdown = 3 - int(elapsed)

    cv2.putText(
        frame,
        str(max(countdown,0)),
        (300,100),
        cv2.FONT_HERSHEY_SIMPLEX,
        2,
        (0,255,0),
        3
    )

    if elapsed >= 3:
        register_started = False
        return True

    return False

def face_in_circle(face, center):

    x1, y1, x2, y2 = map(int, face.bbox)

    face_x = (x1 + x2)//2
    face_y = (y1 + y2)//2

    dx = face_x - center[0]
    dy = face_y - center[1]

    distance = (dx*dx + dy*dy) ** 0.5

    return distance < 50
    
def draw_register_circle(frame):

    h, w = frame.shape[:2]

    center = (w//2, h//2)
    radius = 100
    cv2.circle(
        frame,
        center,
        radius,
        (0,255,0),
        2
    )

    return center
def cosine_similarity(a, b):

    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)

    return np.dot(a, b)

def register_face(face, name):

    embedding = face.embedding

    filename = os.path.join(
        "faces",
        f"{name}.npy"
    )

    np.save(filename, embedding)

    print(f"Registered: {filename}")

#load faces----------------------------------------------------------------
def load_registered_faces():

    database = {}

    for file in os.listdir("faces"):

        if file.endswith(".npy"):

            name = file[:-4]

            embedding = np.load(
                os.path.join(
                    "faces",
                    file
                )
            )

            database[name] = embedding

    return database

def recognize_face(
    face,
    database
):

    best_name = "Unknown"
    best_score = 0

    for name, embedding in database.items():

        score = cosine_similarity(
            face.embedding,
            embedding
        )
        if score > best_score:
            best_score = score
            best_name = name

    if best_score > 0.65:
        return best_name, best_score

    return "Unknown", best_score


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
register_started = False
register_start_time = 0
MODE, PERSON_NAME = register_menu()
DATABASE = load_registered_faces()
print("Loading...")
CAMERA_RES = "1080p"
DISPLAY_RES = "360p"
ACCURACY = "medium"
PROCESSOR = "GPU"
DISPLAY_RES = RESOLUTIONS["360p"]
window_name = "Sanjeev Live"
frame_count = 0
fps = 0
frame_idx = 0
FRAME_SKIP = 4  # AI inference every ... frames
cap = setup_camera(CAMERA_RES)
app = load_model(PROCESSOR, ACCURACY)
prev_time = time.time()

faces = []
try:
    while True:
        status, frame = cap.read()
        if not status:
            print("Offline....")
            break
    
        # Mirror image
        frame = cv2.flip(frame, 1)
    
        # Resize once
        display = cv2.resize(
            frame,
            DISPLAY_RES,
            interpolation=cv2.INTER_AREA
        )

        # AI inference
        if frame_idx % FRAME_SKIP == 0:
            faces = app.get(display)
        frame_idx += 1

        #Register
        if MODE == "register":
            ready = registration_process(
                display,
                faces
            )
        
            if ready:
                register_face(
                    faces[0],
                    PERSON_NAME
                )
        
                break
                
        #Recognition --- dont touch now ---
        if MODE == "live":
            for face in faces:
        
                name, score = recognize_face(
                    face,
                    DATABASE
                )
        
                # print(name, score)--------------------- 

        # Draw ------
        draw_faces(
            display,
            faces,
            DATABASE
        )
    
        # FPS counter -----------------------
        prev_time, frame_count, fps = frame_calculation(
            display,
            prev_time,
            frame_count,
            fps
        )
    
        # Show--------------------
        cv2.imshow(window_name, display)
    
        if should_exit(window_name):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Terminated Safely I think")

Detector : buffalo_l
Provider : CUDAExecutionProvider
Accuracy : medium
Frame Inf: 4
Terminated Safely I think


In [11]:
cap.release()
cv2.destroyAllWindows()
print("Terminated Safely")

Terminated Safely
